# NOTEBOOK : 02_player_features.ipynb
# INPUT    : ../../data/processed/player_gw.csv, ../../data/processed/clean_matches.csv
# OUTPUT   : ../../data/processed/player_features.csv

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("../..")
PROCESSED_DIR = ROOT / "data" / "processed"

df = pd.read_csv(PROCESSED_DIR / "player_gw.csv", low_memory=False)
df["kickoff_time"] = pd.to_datetime(df["kickoff_time"], errors="coerce", utc=True)
df["round"] = pd.to_numeric(df["round"], errors="coerce").fillna(0).astype(int)
df["was_home"] = df["was_home"].astype(str).str.lower().isin(["true", "1", "yes"]).astype(int)

# Ensure numeric types for rolling stats
stat_cols = [
    "minutes", "total_points", "bps", "goals_scored",
    "assists", "ict_index", "goals_conceded", "xP",
]
for col in stat_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

# Chronological sort & deduplicate (player_key, season, round)
df = (
    df.sort_values(["player_key", "season", "round", "kickoff_time"])
    .drop_duplicates(subset=["player_key", "season", "round"], keep="last")
    .reset_index(drop=True)
)

In [2]:
# 1. Opponent defensive strength (strictly prior rolling avg goals conceded per team)
opp_default = 1.40  # Premier League average goals conceded/match
matches_path = PROCESSED_DIR / "clean_matches.csv"

if matches_path.exists() and "opponent_team" in df.columns and df["opponent_team"].dtype == object:
    m_df = pd.read_csv(matches_path).dropna(subset=["Full_Time_Home_Goals", "Full_Time_Away_Goals"]).copy()
    m_df["Match_Date"] = pd.to_datetime(m_df["Match_Date"], errors="coerce")

    home_c = m_df[["season", "Match_Date", "Home_Team", "Full_Time_Away_Goals"]].rename(
        columns={"Home_Team": "team_name", "Full_Time_Away_Goals": "conceded"}
    )
    away_c = m_df[["season", "Match_Date", "Away_Team", "Full_Time_Home_Goals"]].rename(
        columns={"Away_Team": "team_name", "Full_Time_Home_Goals": "conceded"}
    )
    team_def = pd.concat([home_c, away_c], ignore_index=True).sort_values(["team_name", "Match_Date"])
    team_def["opp_def_strength"] = (
        team_def.groupby("team_name")["conceded"]
        .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
        .fillna(opp_default)
    )
    opp_map = (
        team_def.groupby(["season", "team_name"])["opp_def_strength"]
        .mean()
        .reset_index()
        .rename(columns={"team_name": "opponent_team", "opp_def_strength": "opponent_defensive_strength"})
    )
    df = df.merge(opp_map, on=["season", "opponent_team"], how="left")
    df["opponent_defensive_strength"] = df["opponent_defensive_strength"].fillna(opp_default)
else:
    # Fallback when opponent_team is numeric FPL ID: use prior rolling goals_conceded
    df["opponent_defensive_strength"] = (
        df.groupby("player_key")["goals_conceded"]
        .transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
        .fillna(opp_default)
    )

In [3]:
# 2. Strictly prior rolling features (last 3 & 5 gameweeks)
roll_targets = ["minutes", "total_points", "bps", "goals_scored", "assists", "ict_index"]
if "xP" in df.columns:
    roll_targets.append("xP")

g = df.groupby("player_key", group_keys=False)
for col in roll_targets:
    shifted = g[col].shift(1)
    df[f"{col}_roll3"] = (
        shifted.groupby(df["player_key"]).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
    )
    df[f"{col}_roll5"] = (
        shifted.groupby(df["player_key"]).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    )

df["prior_games"] = g.cumcount()

In [4]:
# 3. One-hot encode position (GK, DEF, MID, FWD)
pos_dummies = pd.get_dummies(df["position"], prefix="pos", dtype=int)
for col in ["pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]:
    if col not in pos_dummies.columns:
        pos_dummies[col] = 0
df = pd.concat([df, pos_dummies[["pos_GK", "pos_DEF", "pos_MID", "pos_FWD"]]], axis=1)

# 4. Drop cold-start rows (prior_games == 0) and set explicit target
feature_df = df[df["prior_games"] >= 1].copy()
roll_cols = [c for c in feature_df.columns if c.endswith("_roll3") or c.endswith("_roll5")]
feature_df[roll_cols] = feature_df[roll_cols].fillna(0.0)
feature_df["target_points"] = feature_df["total_points"].astype(float)

out_path = PROCESSED_DIR / "player_features.csv"
feature_df.to_csv(out_path, index=False)
print(f"Saved {len(feature_df):,} rows -> {out_path}")
print(f"Rolling features: {len(roll_cols)} | Seasons: {sorted(feature_df['season'].unique())}")

Saved 183,516 rows -> ..\..\data\processed\player_features.csv
Rolling features: 14 | Seasons: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']
